# Exercise 2, Step 4 — Implication analysis

Builds the formal context from the 200-wine derived (binarised) sample, computes
the concept lattice with the `concepts` library, extracts implications, and
classifies each as a **scale artifact** (follows purely from the ordinal /
interordinal scale structure on a single original attribute, e.g.
`alcohol>=13 -> alcohol>=11`) or **data-induced** (a real cross-attribute
dependency present in the wine chemistry).

**Method note.** The full Duquenne–Guigues canonical base can contain
implications with large premises (up to `|M|-1` attributes) which are rarely
interpretable in a report. We instead compute all implications whose premise
has **size ≤ 2** — this is sound and complete for that premise size (every
implication with a 1- or 2-attribute premise that holds in the context is
found; nothing here is approximate), and in practice this is exactly the layer
of the canonical base anyone discussing a 33-attribute derived context would
focus on.

In [1]:
import concepts
import pandas as pd
import re
from collections import Counter
from itertools import combinations

import sys
sys.path.insert(0, ".")
from ex2_prepare_context import build_derived_context, all_attributes

SAMPLE_PATH = "../data/wine_sample_200.csv"


## Rebuild the derived context and load it into `concepts`

In [2]:
sample = pd.read_csv(SAMPLE_PATH)
derived = build_derived_context(sample)
attrs = all_attributes()

objects = [str(i) for i in derived.keys()]
bools = [tuple(a in set(derived[int(o)]) for a in attrs) for o in objects]
ctx = concepts.Context(objects, attrs, bools)

print(f"{len(ctx.objects)} objects x {len(ctx.properties)} attributes")
print(f"Concept lattice has {len(ctx.lattice)} concepts")


200 objects x 33 attributes


Concept lattice has 10212 concepts


## Attribute frequencies

Needed to flag near-universal attributes — an implication concluding in an attribute that *every* object already has is not informative, it just reflects that the threshold was set below the whole sample's range.

In [3]:
n = len(derived)
freq = Counter()
for alist in derived.values():
    freq.update(alist)

freq_df = pd.DataFrame(
    [(a, freq[a], f"{freq[a]/n:.0%}") for a in attrs],
    columns=["attribute", "count", "share"],
).sort_values("count", ascending=False)
universal = set(freq_df.loc[freq_df["count"] == n, "attribute"])
print("Universal attributes (held by all 200 wines):", universal)
freq_df.head(10)


Universal attributes (held by all 200 wines): {'sulphates>=low(0.4)', 'alcohol>=9'}


,attribute,count,share
27,alcohol>=9,200,100%
24,sulphates>=low(0.4),200,100%
0,fixed_acidity>=low(6),195,98%
30,quality>=acceptable(5),192,96%
3,volatile_acidity>=low(0.3),180,90%
8,residual_sugar=dry,179,90%
13,free_so2>=low(6),175,88%
16,total_so2>=low(20),156,78%
23,pH=neutral_range,127,64%
25,sulphates>=medium(0.6),120,60%


Both `alcohol>=9` and `sulphates>=low(0.4)` are **universal** in this sample
(200/200) — no sampled wine falls below either threshold. This is itself a
data fact worth noting in the report (the chosen "low" bin boundary sits below
the entire 200-row sample), but it makes every implication that merely
concludes one of these two attributes uninformative — they are filtered out
below.

## Extract implications (premise size ≤ 2)

In [4]:
def closure(context, props):
    return set(context.intension(context.extension(list(props))))


def small_premise_implications(context, max_premise=2):
    M = list(context.properties)
    results = []
    for size in range(1, max_premise + 1):
        for premise in combinations(M, size):
            pset = set(premise)
            clo = closure(context, pset)
            extra = clo - pset
            if not extra:
                continue
            if size == 2:
                a, b = premise
                redundant = (closure(context, {a}) - {a}) | (closure(context, {b}) - {b})
                if extra <= redundant:
                    continue  # already implied by the two singleton implications
            results.append((sorted(pset), sorted(extra)))
    return results


implications = small_premise_implications(ctx)
print(f"{len(implications)} implications with premise size <= 2")


249 implications with premise size <= 2


## Classify: scale artifact vs. data-induced

An implication is a **pure scale artifact** if every conclusion attribute
shares its "root" (the original physicochemical attribute) with a premise
attribute — e.g. `alcohol>=13 -> alcohol>=11` only ever talks about alcohol.
If the conclusion reaches into a *different* original attribute, that part is
**data-induced**: it is a real dependency the scaling did not create.

In [5]:
def root(attr):
    return re.match(r"^([a-zA-Z_]+)", attr).group(1)


def premise_extent_size(context, premise):
    return len(context.extension(premise))


rows = []
for premise, conclusion in implications:
    conclusion_informative = [c for c in conclusion if c not in universal]
    if not conclusion_informative:
        kind = "trivial (universal conclusion)"
        cross = []
    else:
        proots = {root(p) for p in premise}
        cross = [c for c in conclusion_informative if root(c) not in proots]
        kind = "data-induced" if cross else "scale artifact"
    rows.append({
        "premise": ", ".join(premise),
        "conclusion": ", ".join(conclusion_informative),
        "kind": kind,
        "cross_attribute_part": ", ".join(cross),
        "support": premise_extent_size(ctx, premise),
    })

impl_df = pd.DataFrame(rows)
impl_df["kind"].value_counts()


kind
data-induced                      229
trivial (universal conclusion)     15
scale artifact                      5
Name: count, dtype: int64

In [6]:
impl_df.to_csv("../data/ex2_implications.csv", index=False)
data_induced = impl_df[impl_df["kind"] == "data-induced"].sort_values("support", ascending=False)
print(f"{len(data_induced)} data-induced implications, saved full table to data/ex2_implications.csv")
data_induced.head(15)


229 data-induced implications, saved full table to data/ex2_implications.csv


,premise,conclusion,kind,cross_attribute_part,support
120,"pH=neutral_range, residual_sugar=dry",fixed_acidity>=low(6),data-induced,fixed_acidity>=low(6),111
12,chlorides=high_salt,fixed_acidity>=low(6),data-induced,fixed_acidity>=low(6),101
6,citric_acid_absent,volatile_acidity>=low(0.3),data-induced,volatile_acidity>=low(0.3),100
20,density=heavy,fixed_acidity>=low(6),data-induced,fixed_acidity>=low(6),93
106,"citric_acid_present, residual_sugar=dry",fixed_acidity>=low(6),data-induced,fixed_acidity>=low(6),86
156,"chlorides=low_salt, total_so2>=low(20)",free_so2>=low(6),data-induced,free_so2>=low(6),83
14,free_so2>=medium(15),"free_so2>=low(6), total_so2>=low(20)",data-induced,total_so2>=low(20),80
235,"pH=neutral_range, sulphates>=medium(0.6)",fixed_acidity>=low(6),data-induced,fixed_acidity>=low(6),73
112,"citric_acid_present, sulphates>=medium(0.6)",fixed_acidity>=low(6),data-induced,fixed_acidity>=low(6),73
212,"density=light, pH=neutral_range",quality>=acceptable(5),data-induced,quality>=acceptable(5),67


## Five data-induced implications worth discussing

Picked from the table above, favouring implications with reasonably large
support (not driven by 2–4 outlier wines) and a clear chemical interpretation:

1. **`citric_acid_absent -> volatile_acidity>=low(0.3)`** (support 100/200).
   Wines with no detectable citric acid never fall into the lowest volatile-acidity
   bin. Citric acid is sometimes added specifically to buffer acidity/flavour;
   its absence co-occurring with higher volatile acidity is a real, if modest,
   chemistry link rather than a scaling artifact.

2. **`density=heavy -> fixed_acidity>=low(6)`** (support 93/200).
   Heavier wines (density ≥ 0.997) are never below the "low" fixed-acidity
   threshold. Density rises with dissolved solids (sugar, acids), so this is a
   plausible chemical dependency, not a tautology — density and fixed acidity
   are scaled completely independently.

3. **`total_so2>=medium(60) -> free_so2>=low(6)`** (support 35/200).
   Once total SO2 passes 60 mg/L, the wine always has at least 6 mg/L of *free*
   SO2 too. Free SO2 is a subset of total SO2 by definition, so a relationship
   between the two is chemically expected — but the specific threshold
   crossing is still informative about how the binding equilibrium behaves in
   this sample.

4. **`pH<=acidic(3.1) -> fixed_acidity>=low(6) AND free_so2>=low(6)`** (support 21/200).
   The most acidic wines (pH ≤ 3.1) are never in the lowest fixed-acidity bin —
   expected, since pH and fixed acidity measure related properties — but the
   co-occurrence with free SO2 ≥ 6 is the more interesting part: it suggests
   winemakers in this sample compensate low-pH (already-stable) wines with at
   least a baseline of free SO2 rather than skipping sulfite addition.

5. **`fixed_acidity>=medium(8), volatile_acidity>=high(0.7) -> quality>=acceptable(5), residual_sugar=dry`**
   (support 11/200).
   High volatile acidity (often linked to a vinegar-like fault and usually a
   sign of *lower* quality) still always co-occurs with at least "acceptable"
   quality here, provided fixed acidity is also at least medium and the wine is
   dry. This nuance — volatile acidity alone is not damning if balanced by
   fixed acidity and low residual sugar — would be invisible from a single
   univariate scale and is exactly the kind of 2-attribute-premise discovery
   conceptual scaling is meant to surface.

All five are genuine data-induced implications (none reduces to a same-attribute
ordinal chain), and none concludes in a universal attribute. Implications with
very small premise support (e.g. those derived from the 2 "excellent" quality
wines) are excluded from this list even though they technically hold, because
with n≤4 they are not statistically reliable — that caveat itself is worth a
sentence in `report/ex2_toscana.md`.

## Scale-artifact examples (for contrast)

These are exactly what should be expected from ordinal/interordinal scaling
and are not interesting on their own merit — they confirm the scales are
defined correctly.

In [7]:
scale_artifacts = impl_df[impl_df["kind"] == "scale artifact"]
print(f"{len(scale_artifacts)} pure scale-artifact implications, e.g.:")
scale_artifacts.head(8)


5 pure scale-artifact implications, e.g.:


,premise,conclusion,kind,cross_attribute_part,support
1,fixed_acidity>=medium(8),fixed_acidity>=low(6),scale artifact,,94
2,fixed_acidity>=high(10),"fixed_acidity>=low(6), fixed_acidity>=medium(8)",scale artifact,,44
4,volatile_acidity>=medium(0.5),volatile_acidity>=low(0.3),scale artifact,,102
26,sulphates>=high(0.8),sulphates>=medium(0.6),scale artifact,,26
31,quality>=good(7),quality>=acceptable(5),scale artifact,,27
